# 第 2 章 02：蒙特卡洛近似期望与随机梯度

这一课只回答一个问题：**当真实期望很难直接计算时，怎样只靠随机样本得到可用的近似？** 这就是蒙特卡洛方法。读完后应能解释样本均值、增量更新与随机梯度下降之间的关系。


## 1. 从“算不出平均”开始

设随机变量 $X$ 服从分布 $p(x)$，我们想知道某个量 $f(X)$ 的平均值：

$$
\mathbb{E}_{X\sim p}[f(X)].
$$

如果所有可能的 $X$ 都列得出来，可以逐项求和；若 $X$ 连续，则可以积分。但在游戏、模拟器或大量训练数据中，分布往往很复杂，直接算这个期望不现实。此时我们可以从 $p$ 中随机抽样，观察实际结果。


## 2. 蒙特卡洛：用样本平均近似期望

独立抽取 $n$ 个样本 $x_1,\ldots,x_n$，计算每个样本对应的 $f(x_i)$，再取平均：

$$
\mathbb{E}_{X\sim p}[f(X)]\approx \frac{1}{n}\sum_{i=1}^{n}f(x_i).
$$

人话就是：**不会直接算长期平均，就反复随机试一试，用试出来的平均结果代替它。** 样本越多，样本均值通常越接近真实期望；这是大数定律提供的直觉。

例如掷骰子的平均点数真实为 $3.5$。一次只掷几次，平均可能是 $4$ 或 $2.8$；掷得越来越多，平均会逐步稳定在 $3.5$ 附近。


## 3. 不保存全部样本，也能更新平均值

若已经有前 $t-1$ 个样本的平均值 $q_{t-1}$，看到新样本 $f(x_t)$ 后，可以直接更新：

$$
q_t=\left(1-\frac{1}{t}\right)q_{t-1}+\frac{1}{t}f(x_t).
$$

等价地：

$$
q_t=q_{t-1}+\frac{1}{t}\bigl(f(x_t)-q_{t-1}\bigr).
$$

它的意思是：新平均值等于旧平均值，加上“新样本与旧平均值的差距”的一小部分。这样无需储存所有历史样本。把 $1/t$ 换成学习率 $\alpha_t$，得到常见的随机近似更新：

$$
q_t=q_{t-1}+\alpha_t\bigl(f(x_t)-q_{t-1}\bigr).
$$


## 4. 蒙特卡洛怎样变成随机梯度

训练模型时，目标常是让平均损失尽量小：

$$
\min_w\ \mathbb{E}_{X\sim p}[L(X;w)].
$$

真实梯度是所有数据梯度的期望：

$$
g=\mathbb{E}_{X\sim p}[\nabla_w L(X;w)].
$$

直接计算它很慢，于是随机抽一个小批量 $\tilde{x}_1,\ldots,\tilde{x}_b$，用梯度平均近似：

$$
\tilde g=\frac{1}{b}\sum_{j=1}^{b}\nabla_w L(\tilde{x}_j;w).
$$

再使用

$$
w\leftarrow w-\alpha\tilde g.
$$

这就是小批量随机梯度下降。随机样本的平均梯度是对真实期望梯度的蒙特卡洛近似；因此，随机梯度下降正是蒙特卡洛原理在优化中的一个应用。


## 5. 放到强化学习里

在固定策略 $\pi$ 下，状态动作对 $(s,a)$ 的回报仍然是随机的：同样的局面和动作，之后可能遇到不同情况。动作价值的定义是条件期望：

$$
Q_\pi(s,a)=\mathbb{E}[G_t\mid S_t=s,A_t=a].
$$

蒙特卡洛估计不需要知道环境的完整转移概率：每次在 $s$ 做了 $a$ 后，就记录该时刻直到回合结束的实际回报 $G_t$。收集到 $N(s,a)$ 次这样的经历后，取平均：

$$
Q_\pi(s,a)\approx\frac{1}{N(s,a)}\sum_{i=1}^{N(s,a)}G_t^{(i)}.
$$

关键不是总共走了多少步，而是这个特定组合 $(s,a)$ 被访问了多少次。为了估计 $Q_\pi$，这一步之后应继续按同一个策略 $\pi$ 行动。


## 6. 小结与自检

- 蒙特卡洛：随机抽样，用样本平均近似期望。
- 增量更新：不用保存全部样本，也能持续估计平均值。
- SGD：用小批量的平均梯度近似总体的期望梯度。
- 强化学习的蒙特卡洛估计：用多条完整轨迹的实际回报平均，估计价值。

自检：为什么 $Q_\pi(s,a)$ 的蒙特卡洛样本是“从这一步到回合结束的回报”，而不是只记录这一步奖励？

答：$Q_\pi(s,a)$ 定义的正是从当前时刻开始的累计回报的期望；只记录一步奖励只能估计即时奖励，遗漏了动作对未来的影响。
